# ¿Aporta valor el ML donde falta el registro de urgencias?
## Notebook corregido: evaluación dirigida, referencias sencillas y sensibilidad conjunta

**Resultado de la revisión ya calculada:** no encontramos mejora de error proporcional frente a «siempre 1» en el grupo principal: municipios menores de 10.000 habitantes sin registro de camas. La mediana comparable empata; los dos Poisson no mejoran esa referencia.

| Método | Error logarítmico: pequeños sin registro de camas (69) | Sin registro de camas, cualquier tamaño (125) |
|---|---:|---:|
| Siempre 1 | 0,04018 | 0,09521 |
| Mediana de comparables | 0,04018 | 0,09521 |
| Poisson demográfico | 0,05856 | 0,22908 |
| Poisson con capacidades, el modelo aplicado | 0,04242 | 0,11971 |

La selección interna elige «siempre 1» en los cinco grupos exteriores. **Esto no verifica que exista un consultorio donde falta registro, ni autoriza llenar los vacíos con 1.** No cambiamos el tablero ni sus denominadores.

**Por qué importa este grupo:** corresponde a 146 de las 203 estimaciones aplicadas. Sin exigir tamaño pequeño, la ausencia de registro de camas corresponde a 173 de las 203. Ausencia de registro NO significa ausencia física de camas.

**Qué se ejecutó:** la nueva comparación y 18 escenarios de sensibilidad se calcularon con JavaScript V8 y el motor del repositorio. Se conservan datos y resultados verificables. Las celdas Python de esta revisión se entregan sin ejecutar aquí por el fallo del entorno: no contienen salidas simuladas. Al ejecutarlas contrastan la reproducción con los resultados calculados.

**Qué cambió:** ahora se comparan cuatro métodos sobre los mismos municipios y se selecciona por error en el grupo pertinente. El rendimiento nacional se conserva como contexto. Se muestran estabilidad entre departamentos, el piso 1 y escenarios que recalculan el máximo y el ranking.

**Límite de la evidencia:** esta revisión se diseñó después de examinar resultados de la MISMA muestra. No es un test externo nuevo ni una evaluación de municipios con cero real. Las diferencias pequeñas no prueban por sí mismas inferioridad estadística.


## 1. Preparar el entorno
El entrenamiento se muestra en Python/NumPy. Conservamos Poisson y sus parámetros originales; la revisión compara cuatro candidatos fijos: constante 1, mediana de comparables, Poisson demográfico y Poisson con capacidades. No se vuelve a buscar un modelo grande entre muchas alternativas.

scikit-learn se usa opcionalmente para contrastar el ajuste Poisson. Si faltan dependencias, descomenta la primera línea de la celda siguiente.

Ejecuta de arriba hacia abajo. Las celdas iniciales cargan datos y muestran los resultados archivados de esta revisión; las siguientes vuelven a entrenar y a comprobarlos. El notebook no modifica el tablero.


In [ ]:
# %pip install "numpy>=1.24,<3" "pandas>=2,<4" "matplotlib>=3.7,<4" "scikit-learn>=1.3,<2"
import sys, json, hashlib, io, re, math, platform, warnings
from pathlib import Path
from urllib.request import Request, urlopen
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

plt.rcParams.update({"figure.figsize": (9, 5), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})
AZUL, NARANJA = "#12659a", "#d66a3a"
display(pd.Series({"Python": sys.version.split()[0], "NumPy": np.__version__,
                   "pandas": pd.__version__, "plataforma": platform.platform()}))


## 2. Cargar una versión identificable, no «lo último» sin control
Se intenta leer del repositorio local. El archivo debe coincidir con el **hash Git del archivo de la versión congelada**. Si no coincide, se informa y se descarga esa versión exacta. Si no hay acceso de red, se detiene: no se usa silenciosamente otro corte.

Los JSON de entrada ya contienen agregados municipales, sin contactos personales. Este cuaderno parte de esos agregados; no vuelve a descargar ni auditar nominalmente el registro nacional de prestadores.

**Para decir en la presentación:** «Podemos identificar exactamente los archivos con los que se produjo este resultado».


In [ ]:
REPO = "Practicantepotencia/tablero-terremoto"
COMMIT = "a350ca6591174542e76b08d108e4d4e15c629b4c"
RAW = f"https://raw.githubusercontent.com/{REPO}/{COMMIT}/"
GIT_BLOBS = {
    "experimentos/ml_salud/entrada.json": "989bac0dcbf1e89d778c7494adc0cfb4e66165b4",
    "experimentos/ml_salud/resultados.json": "0d0dabd32ee5f6fdfac59bbb7a1f656b945d9e2f",
    "experimentos/ml_salud/validacion.csv": "d14913c0608b6898e5d21ef362939d1ed75a6884",
    "experimentos/ml_salud/impacto_aplicacion_urgencias.json": "95efc6f0cae6bf0deaacb163dd9ab0bdc5fb8b3d",
    "index.html": "98592fadffd7fb9d7b821b7420c8948be6c4aef1"
}
_cache = {}
trazabilidad = []

def git_blob_sha(data):
    return hashlib.sha1(b"blob " + str(len(data)).encode() + b"\0" + data).hexdigest()

def leer_version(path):
    if path in _cache:
        return _cache[path]
    esperado = GIT_BLOBS[path]
    contenido, procedencia = None, None
    for raiz in [Path.cwd(), *Path.cwd().parents]:
        local = raiz / path
        if local.is_file():
            candidato = local.read_bytes()
            if git_blob_sha(candidato) == esperado:
                contenido, procedencia = candidato, str(local)
                break
            print(f"Se omite copia local distinta del corte: {local}")
            break
    if contenido is None:
        url = RAW + path
        try:
            with urlopen(Request(url, headers={"User-Agent": "Notebook-Salud-Reproducible"}), timeout=120) as respuesta:
                contenido = respuesta.read()
            procedencia = url
        except Exception as exc:
            raise RuntimeError(
                f"No se pudo cargar {path}. Se necesita acceso al repositorio, "
                f"o una copia local exacta del commit {COMMIT}."
            ) from exc
    obtenido = git_blob_sha(contenido)
    if obtenido != esperado:
        raise ValueError(f"Integridad incorrecta: {path}: {obtenido} != {esperado}")
    _cache[path] = contenido
    trazabilidad.append({"archivo": path, "blob_git": obtenido, "bytes": len(contenido),
                         "origen": procedencia})
    return contenido

entrada = json.loads(leer_version("experimentos/ml_salud/entrada.json"))
archivo_resultados = json.loads(leer_version("experimentos/ml_salud/resultados.json"))
historico = next(r for r in archivo_resultados["results"] if r["target"]["id"] == "urgencias")
validacion_archivada = pd.read_csv(
    io.BytesIO(leer_version("experimentos/ml_salud/validacion.csv")), dtype={"codigo": str}
)
display(pd.DataFrame(trazabilidad))


In [ ]:
# Artefactos de la revisión fijados a otro commit, sin cambiar el corte de los datos originales.
COMMIT_REVISION = "01cfb4916e52357f61db4eba3c961cf8fa7eeaf2"
BLOBS_REVISION = {
    "experimentos/ml_salud/revision_faltantes/comparacion.json": "07a083971766f94b9b7180ea98955a1dab0f942b",
    "experimentos/ml_salud/revision_faltantes/sensibilidad.json": "957641c62523d2a3b199b0cf74c218ec1cdf1b45"
}

def leer_revision(path):
    esperado = BLOBS_REVISION[path]
    contenido, origen = None, None
    for raiz in [Path.cwd(), *Path.cwd().parents]:
        local = raiz / path
        if local.is_file():
            candidato = local.read_bytes()
            if git_blob_sha(candidato) == esperado:
                contenido, origen = candidato, str(local)
            break
    if contenido is None:
        origen = f"https://raw.githubusercontent.com/{REPO}/{COMMIT_REVISION}/{path}"
        with urlopen(Request(origen, headers={"User-Agent": "Notebook-Salud-Reproducible"}), timeout=120) as resp:
            contenido = resp.read()
    if git_blob_sha(contenido) != esperado:
        raise ValueError(f"El archivo de revisión no coincide: {path}")
    trazabilidad.append({"archivo": path, "blob_git": esperado, "bytes": len(contenido), "origen": origen})
    return json.loads(contenido)

revision = leer_revision("experimentos/ml_salud/revision_faltantes/comparacion.json")
sensibilidad_archivada = leer_revision("experimentos/ml_salud/revision_faltantes/sensibilidad.json")
metodos_revision = revision["protocol"]["candidates"]
metricas_archivadas_revision = pd.DataFrame(revision["metrics"])
display(metricas_archivadas_revision.loc[
    metricas_archivadas_revision["group"].isin(["primary_small_no_beds", "without_beds"])
    & metricas_archivadas_revision["method"].isin(metodos_revision),
    ["group", "method", "n", "mean_absolute_log_error", "mae", "r2"]
])
print("Tabla archivada de la nueva corrida JavaScript; aún no es la ejecución Python de abajo.")


## 3. Qué mide cada fuente y de qué año es

| Información | Fuente de origen | Referencia |
|---|---|---|
| Consultorios de urgencias, consulta externa y camas generales | Ministerio de Salud / REPS | 5 de noviembre de 2022 |
| Población total municipal | Proyecciones DANE | 2026 |
| IPM municipal | DANE, fuente censal | 2018 |
| Centros de salud afectados | Inventario PNUD del tablero | Captura del 11 de septiembre de 2026 |

El extracto REPS se conservó a través de un espejo inmutable; la URL y los hashes están en la entrada. **Fecha de publicación del archivo no equivale a fecha del registro.** Las fechas de captura del tablero tampoco prueban la fecha de cada observación de daño.

Esta combinación temporal permite un ensayo exploratorio; no acredita capacidad operativa en 2026.


In [ ]:
display(pd.DataFrame([
    {"variable": "Capacidad sanitaria", "referencia": entrada["target_date"],
     "fuente": entrada["sources"]["capacity"]["url"],
     "extracto": entrada["sources"]["capacity"]["mirror_url"]},
    {"variable": "Población", "referencia": entrada["population_year"],
     "fuente": entrada["sources"]["population"]["url"]},
    {"variable": "IPM", "referencia": entrada["ipm_year"],
     "fuente": entrada["sources"]["ipm"]["url"]},
]))

df = pd.DataFrame(entrada["rows"])
df["code"] = df["code"].astype(str).str.zfill(5)
for j, nombre in enumerate(["consulta_externa", "urgencias", "camas_generales"]):
    df[nombre] = df["capacity_2022"].apply(lambda v: v[j]).astype(float)

assert len(df) == 1122 and df["code"].is_unique
assert df["code"].str.fullmatch(r"\d{5}").all()
assert np.isfinite(df["population_2026"]).all() and df["population_2026"].gt(0).all()
for c in ["consulta_externa", "urgencias", "camas_generales"]:
    assert df[c].dropna().gt(0).all(), f"Revisar ceros/negativos en {c}"
display(df[["code", "municipality", "department", "population_2026",
            "ipm_2018", "consulta_externa", "urgencias", "camas_generales"]].head(10))


## 4. Cobertura antes de entrenar
Un **faltante no es cero**. No entrenamos con etiquetas inventadas ni repetimos el mismo municipio por cada captura diaria.

En urgencias hay 901 etiquetas positivas y 221 ausencias de registro. Aprender sólo con positivos no permite aprender cuándo el servicio realmente no existe.


In [ ]:
cobertura = pd.DataFrame([
    {"denominador": c, "observados_positivos": int(df[c].notna().sum()),
     "faltantes": int(df[c].isna().sum()), "ceros_observados": int(df[c].eq(0).sum())}
    for c in ["consulta_externa", "urgencias", "camas_generales"]
])
display(cobertura)
observados = df.loc[df["urgencias"].notna()].copy().reset_index(drop=True)
faltantes = df.loc[df["urgencias"].isna()].copy().reset_index(drop=True)
assert len(observados) == 901 and len(faltantes) == 221
assert observados["population_2026"].lt(50000).sum() == 750

fig, ax = plt.subplots()
ax.bar(cobertura["denominador"], cobertura["observados_positivos"], color=AZUL, label="Registro positivo")
ax.bar(cobertura["denominador"], cobertura["faltantes"],
       bottom=cobertura["observados_positivos"], color="#dbe4ea", label="Sin registro utilizable")
ax.set(ylabel="Municipios", title="Cobertura nacional de capacidad sanitaria")
ax.legend(); plt.tight_layout(); plt.show()


### ¿Se parecen la evaluación y el lugar donde imputamos?
El R² nacional no basta: los observados y los faltantes tienen distribuciones diferentes. La siguiente tabla lo cuantifica.

El grupo principal se define mediante datos conocidos ANTES de predecir: población menor de 10.000 y falta de registro de camas. El grupo 1–5 consultorios se mostrará más adelante sólo como diagnóstico, porque su valor verdadero es desconocido en los faltantes.

En los 69 observados del grupo principal, 65 tienen una unidad y 4 tienen dos. Son 11 departamentos, con muy distintos tamaños de muestra; se debe mostrar esa concentración.


In [ ]:
def grupo_principal(frame):
    return frame["camas_generales"].isna() & frame["population_2026"].lt(10000)

ids_elegibles = {r["code"] for r in historico["predictions"] if r["exploratory_supported"]}
elegibles = df.loc[df["code"].isin(ids_elegibles)]
distribucion = pd.DataFrame([
    {"grupo": etiqueta, "n": len(g), "mediana_poblacion": g["population_2026"].median(),
     "con_registro_camas": int(g["camas_generales"].notna().sum()),
     "pequenos_sin_registro_camas": int(grupo_principal(g).sum())}
    for etiqueta, g in [("Observados", observados), ("Todos los faltantes", faltantes),
                        ("Estimaciones aplicadas", elegibles)]
])
display(distribucion)
assert int(grupo_principal(observados).sum()) == 69
assert int(grupo_principal(elegibles).sum()) == 146
assert int(elegibles["camas_generales"].isna().sum()) == 173

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].bar(distribucion["grupo"], distribucion["mediana_poblacion"], color=AZUL)
axs[0].set(ylabel="Habitantes", title="Tamaño municipal: mediana")
axs[1].bar(distribucion["grupo"], 100*distribucion["con_registro_camas"]/distribucion["n"], color=NARANJA)
axs[1].set(ylabel="% con registro de camas", title="Disponibilidad de información")
for ax in axs: ax.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()


## 5. Construir los predictores sin filtrar información de validación
Para urgencias usamos siete columnas:

1. \(\log(1+\text{población})\).
2. IPM.
3. Bandera de IPM ausente.
4. \(\log(1+\text{consulta externa})\), o un 0 técnico si falta.
5. Bandera de consulta externa ausente.
6. \(\log(1+\text{camas generales})\), o un 0 técnico si falta.
7. Bandera de camas ausentes.

El 0 técnico con bandera **no es una observación de capacidad cero**. Las medianas, medias y desviaciones se calculan de nuevo **sólo con cada conjunto de entrenamiento**.

No entran daños, puntajes PNUD/UNGRD, el índice final ni el propio valor de urgencias. Tampoco usamos otras capacidades predichas para predecir ésta.


In [ ]:
NOMBRES_X = [
    "log1p(población 2026)", "IPM 2018", "IPM faltante",
    "log1p(consulta externa 2022)", "consulta externa faltante",
    "log1p(camas generales 2022)", "camas generales faltantes"
]

def variables(frame, enriquecido=True):
    columnas = [
        np.log1p(frame["population_2026"].to_numpy(dtype=float)),
        frame["ipm_2018"].to_numpy(dtype=float),
        frame["ipm_2018"].isna().to_numpy(dtype=float),
    ]
    if enriquecido:
        for c in ["consulta_externa", "camas_generales"]:
            columnas.extend([
                np.log1p(frame[c].fillna(0).to_numpy(dtype=float)),
                frame[c].isna().to_numpy(dtype=float),
            ])
    return np.column_stack(columnas)

def ajustar_preproceso(frame, enriquecido=True):
    X = variables(frame, enriquecido)
    medianas = np.array([
        np.median(col[np.isfinite(col)]) if np.isfinite(col).any() else 0.
        for col in X.T
    ])
    completo = np.where(np.isfinite(X), X, medianas)
    medias = completo.mean(axis=0)
    desv = completo.std(axis=0, ddof=0)
    desv = np.where(desv == 0, 1., desv)
    return {"med": medianas, "avg": medias, "sd": desv}

def transformar(frame, pre, enriquecido=True):
    X = variables(frame, enriquecido)
    return (np.where(np.isfinite(X), X, pre["med"]) - pre["avg"]) / pre["sd"]

display(pd.DataFrame(variables(observados), columns=NOMBRES_X).head())


## 6. Entrenar Poisson: qué función se minimiza
Para \(z_i\), los predictores estandarizados del municipio \(i\):

\[
\eta_i=\beta_0+z_i^\top\beta,\qquad \mu_i=\exp(\eta_i)
\]

\[
L(\beta_0,\beta)=\frac1n\sum_i\left[\exp(\eta_i)-y_i\eta_i\right]
+\frac{\alpha}{2}\sum_j\beta_j^2,\qquad \alpha=1
\]

El término final es la regularización L2. El intercepto no se penaliza. Se omiten constantes que no cambian el ajuste.

La predicción publicada se recorta después:

\[
\widehat B_i=\max(1,\mu_i).
\]

**Importante:** el piso 1 no forma parte de la optimización; es una restricción posterior del ensayo de positivos. No hemos ajustado un modelo que distinga ceros reales ni una distribución Poisson truncada formal.

La celda muestra Newton con búsqueda de paso, como en el código original. NumPy resuelve el sistema lineal.


In [ ]:
def ajustar_poisson(X, y, alpha=1.):
    y = np.asarray(y, dtype=float)
    X1 = np.column_stack([np.ones(len(y)), X])
    n, p = X1.shape
    beta = np.zeros(p)
    beta[0] = np.log(y.mean())
    penalizacion = np.r_[0., np.full(p - 1, alpha)]

    def perdida(b):
        eta = X1 @ b
        if not np.isfinite(eta).all() or np.max(eta) > 30:
            return np.inf
        return np.mean(np.exp(eta) - y * eta) + alpha * np.sum(b[1:]**2) / 2

    convergio = False
    for it in range(80):
        mu = np.exp(X1 @ beta)
        gradiente = X1.T @ (mu - y) / n + penalizacion * beta
        hessiano = (X1.T * mu) @ X1 / n + np.diag(penalizacion)
        if np.max(np.abs(gradiente)) < 1e-7:
            convergio = True
            break
        paso = np.linalg.solve(hessiano, gradiente)
        previo, tasa, aceptado = perdida(beta), 1., False
        for _ in range(35):
            candidato = beta - tasa * paso
            if perdida(candidato) <= previo - 1e-4 * tasa * float(gradiente @ paso):
                beta, aceptado = candidato, True
                break
            tasa /= 2
        if not aceptado:
            convergio = np.max(np.abs(paso)) < 1e-6
            break
        if np.max(np.abs(tasa * paso)) < 1e-8:
            convergio = True
            break
    return {"kind": "poisson", "beta": beta, "converged": bool(convergio),
            "iterations": it + 1, "alpha": alpha}

# Prueba analítica: sin información predictiva y con y constante, se recupera su media.
prueba = ajustar_poisson(np.zeros((40, 2)), np.full(40, 3.))
assert prueba["converged"]
np.testing.assert_allclose(np.exp(prueba["beta"][0]), 3., rtol=1e-8)
np.testing.assert_allclose(prueba["beta"][1:], 0., atol=1e-8)
print("Prueba analítica de Poisson: correcta.")


## 7. Cuatro métodos, la misma evaluación

- **Siempre 1:** referencia sencilla, no una imputación autorizada.
- **Mediana de comparables:** mediana observada de entrenamiento con igual disponibilidad de camas y la misma banda de población. Bandas fijas: <5 mil, 5–10 mil, 10–20 mil, 20–50 mil, 50–100 mil, ≥100 mil. Si no hay ejemplos, usa la misma disponibilidad de camas; si tampoco hay, todo entrenamiento.
- **Poisson demográfico:** población e IPM, incluidas sus banderas de ausencia.
- **Poisson con capacidades:** modelo actual, con las otras capacidades observadas.

Los modelos se ajustan con todos los municipios del conjunto de entrenamiento. **La elección se decide por el error en el grupo principal dentro de la validación interna**, no por la evaluación exterior ni por UNGRD. Empates favorecen el método más sencillo en el orden de arriba. No se ajustan hiperparámetros nuevos.

Los grupos de validación externa siguen siendo los departamentos originales. Las reglas del grupo principal se fijan para esta revisión, que se declara exploratoria y posterior al análisis original.


In [ ]:
CANDIDATOS = [
    {"id": "siempre_1", "family": "constant", "enriched": False},
    {"id": "mediana_comparables", "family": "comparable_median", "enriched": False},
    {"id": "poisson_demografia", "family": "poisson", "enriched": False},
    {"id": "poisson_capacidades", "family": "poisson", "enriched": True},
]
METODOS = [c["id"] for c in CANDIDATOS]
LIMITES_COMPARABLES = np.array([5000, 10000, 20000, 50000, 100000])

def entrenar(frame, config):
    y = frame["urgencias"].to_numpy(dtype=float)
    assert np.isfinite(y).all() and (y > 0).all()
    if config["family"] == "constant":
        return {"config": config, "model": {"kind": "constant"}}
    if config["family"] == "comparable_median":
        return {"config": config, "model": {"kind": "comparable_median", "train": frame.copy()}}
    pre = ajustar_preproceso(frame, config["enriched"])
    X = transformar(frame, pre, config["enriched"])
    modelo = ajustar_poisson(X, y, alpha=1.)
    if not modelo["converged"]:
        warnings.warn("Poisson no declaró convergencia: revisar antes de presentar.")
    return {"config": config, "scaler": pre, "model": modelo}

def predecir(ajuste, frame, piso=True):
    modelo = ajuste["model"]
    if modelo["kind"] == "constant":
        return np.ones(len(frame))
    if modelo["kind"] == "comparable_median":
        train = modelo["train"]
        bandas_train = np.searchsorted(LIMITES_COMPARABLES, train["population_2026"].to_numpy(), side="right")
        valores = []
        for _, row in frame.iterrows():
            banda = np.searchsorted(LIMITES_COMPARABLES, row["population_2026"], side="right")
            mismas_camas = train["camas_generales"].notna().eq(pd.notna(row["camas_generales"]))
            pool = train.loc[mismas_camas & (bandas_train == banda), "urgencias"]
            if pool.empty:
                pool = train.loc[mismas_camas, "urgencias"]
            if pool.empty:
                pool = train["urgencias"]
            valores.append(float(pool.median()))
        return np.array(valores)
    X = transformar(frame, ajuste["scaler"], ajuste["config"]["enriched"])
    pred = np.exp(modelo["beta"][0] + X @ modelo["beta"][1:])
    return np.maximum(1., pred) if piso else pred


## 8. Métricas: qué se elige y qué se informa
La selección final minimiza el error logarítmico absoluto medio:

\[
E_{\log}=\frac1n\sum_i|\log(\widehat B_i)-\log(B_i)|.
\]

Equivocarse por un factor 2 pesa igual en un municipio pequeño o grande. Como usamos daño/base, un error multiplicativo en la base se transmite al cociente cuando el daño es positivo. **Eso no garantiza conservar el ranking**, porque un máximo también puede cambiar.

Se informan además MAE, RMSE, R² predictivo y error porcentual mediano. R² aquí es \(1-\mathrm{SSE}/\mathrm{SST}\), no la correlación al cuadrado.

**Transparencia:** el primer ensayo se seleccionó por MAE nacional; después de explorar resultados se cambió al error proporcional. Esta es validación exploratoria, no una confirmación externa intacta. El archivo `ensayo_inicial_mae.json` conserva el primer ensayo.


In [ ]:
def metricas(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    assert len(y) == len(p) and len(y) > 0
    assert np.isfinite(y).all() and np.isfinite(p).all() and (y > 0).all() and (p > 0).all()
    ae = np.abs(p-y)
    sst = np.sum((y-y.mean())**2)
    return {"n": len(y), "mae": float(ae.mean()),
            "rmse": float(np.sqrt(np.mean((y-p)**2))),
            "r2": float(1-np.sum((y-p)**2)/sst) if sst > 0 else np.nan,
            "mean_absolute_log_error": float(np.abs(np.log(np.maximum(p, 1))-np.log(y)).mean()),
            "median_absolute_percentage_error": float(np.median(ae/y*100)),
            "within_one_fraction": float(np.mean(ae <= 1))}

assert np.isclose(metricas([1, 100], [2, 200])["mean_absolute_log_error"], np.log(2))
assert metricas([1, 2, 3], [1, 2, 3])["r2"] == 1
assert metricas([1, 2, 3], [3, 2, 1])["r2"] < 0
print("Comprobaciones de métricas: correctas.")


## 9. Separar departamentos completos y seleccionar para el uso pertinente
Se mantienen los cinco grupos de departamentos del experimento original. Ningún departamento aparece simultáneamente en entrenamiento y evaluación.

Dentro de cada entrenamiento se hacen tres particiones internas por departamento. Los cuatro métodos se ajustan con los municipios de entrenamiento disponibles; la elección usa sólo el error en los pequeños sin registro de camas que quedaron reservados dentro de esa validación interna.

Después se reentrena el elegido sin los departamentos exteriores reservados. La comparación de los cuatro métodos en los 901 observados se guarda por separado. Elegir una constante para el grupo principal no significa recomendarla para todo el país.

Esta revisión reutiliza datos que ya habíamos examinado: la separación evita fuga de etiquetas durante el ajuste, pero no convierte la revisión exploratoria en una prueba externa nueva.


In [ ]:
def particiones(frame, k):
    conteos = Counter(frame["department"])
    if len(conteos) < k:
        raise ValueError("No hay suficientes departamentos.")
    grupos = [{"n": 0, "departments": []} for _ in range(k)]
    for dep, n in sorted(conteos.items(), key=lambda x: (-x[1], x[0])):
        j = min(range(k), key=lambda i: grupos[i]["n"])
        grupos[j]["n"] += n
        grupos[j]["departments"].append(dep)
    return grupos

externos = particiones(observados, 5)
assert len(set(d for g in externos for d in g["departments"])) == observados["department"].nunique()
for nuevo, guardado in zip(externos, historico["folds"]):
    assert nuevo["departments"] == guardado["held_out_departments"]
display(pd.DataFrame([
    {"grupo": i+1, "evaluación": g["n"], "entrenamiento": len(observados)-g["n"],
     "departamentos reservados": ", ".join(g["departments"])}
    for i, g in enumerate(externos)
]))

def seleccionar(frame, k=3):
    resultados = []
    for config in CANDIDATOS:
        reales, predichos = [], []
        for g in particiones(frame, k):
            mascara = frame["department"].isin(g["departments"])
            train = frame.loc[~mascara]
            test = frame.loc[mascara & grupo_principal(frame)]
            assert set(train["department"]).isdisjoint(test["department"])
            ajuste = entrenar(train, config)
            reales.extend(test["urgencias"].to_numpy())
            predichos.extend(predecir(ajuste, test))
        if not reales:
            raise ValueError("No hay observaciones del grupo principal en la validación interna.")
        resultados.append({"id": config["id"], **metricas(reales, predichos)})
    tabla = pd.DataFrame(resultados).sort_values("mean_absolute_log_error", kind="stable")
    ganador = next(c for c in CANDIDATOS if c["id"] == tabla.iloc[0]["id"])
    return ganador, tabla


## 10. Reentrenar y comparar fuera de muestra
Cada municipio recibe cuatro predicciones de modelos que no vieron su departamento. En paralelo, la selección interna escoge un método utilizando sólo los observados pequeños sin registro de camas del entrenamiento.

Para conservar trazabilidad, **la columna `predicho` sigue significando Poisson con capacidades, el modelo aplicado originalmente**. La nueva decisión se guarda por separado en `seleccion_interna` y `prediccion_seleccionada`. No se aplica automáticamente a faltantes.


In [ ]:
filas_oof, decisiones, tablas_internas = [], [], []
for i, g in enumerate(externos, start=1):
    mascara = observados["department"].isin(g["departments"])
    train, test = observados.loc[~mascara], observados.loc[mascara]
    ganador, tabla = seleccionar(train, k=3)
    tablas_internas.append(tabla.assign(grupo_externo=i))
    salida = test[["code", "municipality", "department", "population_2026", "urgencias", "camas_generales"]].copy()
    salida = salida.rename(columns={"urgencias": "observado"})
    for config in CANDIDATOS:
        ajuste = entrenar(train, config)
        salida[config["id"]] = predecir(ajuste, test)
    salida["predicho"] = salida["poisson_capacidades"]  # modelo aplicado, no nuevo ganador
    salida["grupo"] = i
    salida["modelo"] = "poisson_capacidades"
    salida["seleccion_interna"] = ganador["id"]
    salida["prediccion_seleccionada"] = salida[ganador["id"]]
    filas_oof.append(salida)
    decisiones.append({"grupo": i, "seleccion": ganador["id"], "n_train": len(train),
                       "n_test": len(test), "n_principal_test": int(grupo_principal(test).sum())})
    print(f"Grupo {i}/5: selección {ganador['id']}; {len(test)} municipios evaluados.", flush=True)
oof = pd.concat(filas_oof, ignore_index=True)
assert len(oof) == 901 and oof["code"].is_unique
display(pd.DataFrame(decisiones))
display(pd.concat(tablas_internas, ignore_index=True)[
    ["grupo_externo", "id", "n", "mean_absolute_log_error", "mae", "r2"]])


## 11. Comprobar reproducción y mostrar primero el uso pertinente
Se comprueban las cuatro predicciones contra la revisión ejecutada en JavaScript y el modelo aplicado contra sus 901 predicciones originales. No se presentan cifras archivadas como si fueran una ejecución Python.

La evaluación principal tiene sólo 69 municipios y 11 departamentos; los cinco grupos exteriores contienen 2, 29, 24, 8 y 6 ejemplos principales. Es una limitación importante de la fuerza de la evidencia.


In [ ]:
arch = validacion_archivada.loc[validacion_archivada["denominador"].eq("urgencias")].copy()
arch = arch.rename(columns={"codigo": "code",
                            "predicho_sin_ver_departamento": "predicho_archivado"})
comparacion_oof = oof.merge(arch[["code", "predicho_archivado", "modelo"]],
                           on="code", how="inner", validate="one_to_one", suffixes=("", "_archivado"))
assert len(comparacion_oof) == 901
assert comparacion_oof["modelo"].eq(comparacion_oof["modelo_archivado"]).all()
np.testing.assert_allclose(comparacion_oof["predicho"], comparacion_oof["predicho_archivado"],
                           rtol=1e-5, atol=1e-5)
print("Máxima diferencia absoluta Python frente al registro JavaScript:",
      float(np.max(np.abs(comparacion_oof["predicho"]-comparacion_oof["predicho_archivado"]))))


nuevas_archivadas = pd.DataFrame([
    {"code": r["code"], **r["predictions"], "seleccion_interna": r["selected_id"]}
    for r in revision["oof"]
]).set_index("code")
actuales = oof.set_index("code").loc[nuevas_archivadas.index]
for metodo in METODOS:
    np.testing.assert_allclose(actuales[metodo], nuevas_archivadas[metodo], rtol=1e-5, atol=1e-5)
assert actuales["seleccion_interna"].eq(nuevas_archivadas["seleccion_interna"]).all()
print("Las cuatro predicciones y la selección coinciden con la revisión guardada.")

grupos_evaluacion = {
    "primary_small_no_beds": oof["camas_generales"].isna() & oof["population_2026"].lt(10000),
    "without_beds": oof["camas_generales"].isna(),
    "with_beds": oof["camas_generales"].notna(),
    "all": pd.Series(True, index=oof.index),
    "observed_1_to_5_diagnostic": oof["observado"].le(5),
}
resumen_revision = pd.DataFrame([
    {"grupo": nombre, "metodo": metodo, **metricas(oof.loc[mascara, "observado"], oof.loc[mascara, metodo])}
    for nombre, mascara in grupos_evaluacion.items() for metodo in METODOS
])
display(resumen_revision.loc[resumen_revision["grupo"].isin(["primary_small_no_beds", "without_beds"]),
    ["grupo", "metodo", "n", "mean_absolute_log_error", "mae", "r2", "within_one_fraction"]])
for _, r in resumen_revision.iterrows():
    ref = next(x for x in revision["metrics"] if x["group"] == r["grupo"] and x["method"] == r["metodo"])
    np.testing.assert_allclose(r["mean_absolute_log_error"], ref["mean_absolute_log_error"], rtol=1e-5, atol=1e-7)

fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for ax, grupo, titulo in zip(axs, ["primary_small_no_beds", "without_beds"],
                            ["Pequeños sin registro de camas (69)", "Sin registro de camas (125)"]):
    g = resumen_revision.loc[resumen_revision["grupo"].eq(grupo)]
    ax.barh(g["metodo"], g["mean_absolute_log_error"], color=[NARANJA, "#999999", "#7ba7c7", AZUL])
    ax.set(xlabel="Error logarítmico medio — menor es mejor", title=titulo)
    ax.invert_yaxis()
plt.tight_layout(); plt.show()


### ¿La diferencia se sostiene entre departamentos?
Una media puede esconder concentración del error. Comparamos **en los mismos municipios** el error de cada método menos el error de «siempre 1». Negativo favorece el método; cero es empate; positivo favorece la constante.

Mostramos diferencias por grupo exterior y una banda descriptiva obtenida al remuestrear departamentos completos 2.000 veces. La banda conserva el emparejamiento entre métodos, pero **no es validación externa, ni intervalo de predicción, ni garantía para los faltantes**. El protocolo fue diseñado después de revisar estos datos y se comparan varios métodos: no se usa la banda como prueba confirmatoria.

En el grupo principal, la pequeña desventaja del modelo aplicado proviene de dos municipios del mismo departamento; no debe presentarse como una inferioridad amplia y demostrada.


In [ ]:
por_grupo_exterior = pd.DataFrame([
    {"grupo": nombre, "fold": fold, "metodo": metodo,
     **metricas(g["observado"], g[metodo])}
    for nombre in ["primary_small_no_beds", "without_beds"]
    for fold, g in oof.loc[grupos_evaluacion[nombre]].groupby("grupo")
    for metodo in METODOS
])
display(por_grupo_exterior.loc[por_grupo_exterior["grupo"].eq("primary_small_no_beds"),
    ["fold", "metodo", "n", "mean_absolute_log_error"]])

def bootstrap_departamentos(g, metodo, repeticiones=2000):
    diferencias = np.abs(np.log(g[metodo]/g["observado"])) - np.abs(np.log(1/g["observado"]))
    agrupado = pd.DataFrame({"dep": g["department"], "delta": diferencias}).groupby("dep", sort=True)["delta"].agg(["sum", "count"])
    sumas, ns = agrupado["sum"].to_numpy(), agrupado["count"].to_numpy()
    seed = 20260915
    resultados = []
    for _ in range(repeticiones):
        total, n = 0., 0
        for _ in range(len(agrupado)):
            seed = (1664525*seed + 1013904223) & 0xffffffff
            j = int((seed / 4294967296) * len(agrupado))
            total += sumas[j]; n += ns[j]
        resultados.append(total/n)
    medias = sumas/ns
    return {"delta": float(diferencias.mean()), "inferior": float(np.quantile(resultados, .025)),
            "superior": float(np.quantile(resultados, .975)), "departamentos": len(agrupado),
            "mejora": int(np.sum(medias < -1e-12)), "empata": int(np.sum(np.abs(medias) <= 1e-12)),
            "empeora": int(np.sum(medias > 1e-12))}

estabilidad = pd.DataFrame([
    {"grupo": nombre, "metodo": metodo, **bootstrap_departamentos(oof.loc[grupos_evaluacion[nombre]], metodo)}
    for nombre in ["primary_small_no_beds", "without_beds"] for metodo in METODOS if metodo != "siempre_1"
])
display(estabilidad)
for _, r in estabilidad.iterrows():
    ref = next(x for x in revision["comparisons"] if x["group"] == r["grupo"] and x["method"] == r["metodo"])
    np.testing.assert_allclose([r["delta"], r["inferior"], r["superior"]],
        [ref["delta"], *ref["bootstrap_department_percentile_95"]], rtol=1e-4, atol=1e-6)

g = estabilidad.loc[estabilidad["grupo"].eq("primary_small_no_beds")].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(9, 4))
ax.errorbar(g["delta"], np.arange(len(g)),
            xerr=np.vstack([np.maximum(0, g["delta"]-g["inferior"]), np.maximum(0, g["superior"]-g["delta"])]),
            fmt="o", color=AZUL, capsize=4)
ax.axvline(0, color=NARANJA, linestyle="--")
ax.set_yticks(np.arange(len(g)), g["metodo"])
ax.set(xlabel="Error del método − error de «siempre 1»", title="Grupo principal: diferencia y banda descriptiva")
plt.tight_layout(); plt.show()


## 12. Contexto nacional: lo que el R² global no representa
Ahora, después de los resultados pertinentes, mostramos la visión nacional del **Poisson aplicado originalmente**. La diagonal es acierto; encima, sobreestimación; debajo, subestimación.

Se añaden las referencias sencillas y el grupo de 1–5 consultorios. Este último se define por la etiqueta real y es sólo diagnóstico, no una regla utilizable cuando el denominador es desconocido.

R² bajo o negativo en grupos casi constantes puede coexistir con errores absolutos pequeños. Se deben leer juntos R², MAE y la comparación con constantes. ±1 consultorio es una tolerancia muy amplia para municipios con una sola unidad; no demuestra precisión suficiente del cociente daño/base.


In [ ]:
oof["error"] = oof["predicho"] - oof["observado"]
oof["error_absoluto"] = oof["error"].abs()
oof["error_porcentual"] = 100 * oof["error"] / oof["observado"]

fig, axs = plt.subplots(1, 2, figsize=(13, 5))
axs[0].scatter(oof["observado"], oof["predicho"], alpha=.45, s=24, color=AZUL)
lo = min(oof["observado"].min(), oof["predicho"].min())
hi = max(oof["observado"].max(), oof["predicho"].max())
axs[0].plot([lo, hi], [lo, hi], "--", color=NARANJA, label="Predicción perfecta")
axs[0].set(xscale="log", yscale="log", xlabel="Consultorios observados (REPS 2022)",
           ylabel="Consultorios predichos fuera de muestra", title="Real frente a predicho")
axs[0].legend()
axs[1].scatter(oof["observado"], oof["error"], alpha=.45, s=24, color=AZUL)
axs[1].axhline(0, color=NARANJA, linestyle="--")
axs[1].set(xscale="log", xlabel="Consultorios observados", ylabel="Predicho − observado",
           title="Error con signo, en consultorios")
plt.tight_layout(); plt.show()

limites = [0, 5000, 10000, 20000, 50000, 100000, np.inf]
etiquetas = ["<5 mil", "5–10 mil", "10–20 mil", "20–50 mil", "50–100 mil", "≥100 mil"]
oof["grupo_poblacion"] = pd.cut(oof["population_2026"], limites, labels=etiquetas, right=False)
por_tamano = pd.DataFrame([
    {"poblacion": str(nombre), **metricas(g["observado"], g["predicho"])}
    for nombre, g in oof.groupby("grupo_poblacion", observed=True)
])
fig, ax = plt.subplots()
ax.bar(por_tamano["poblacion"], por_tamano["mae"], color=AZUL)
ax.set(xlabel="Población proyectada 2026", ylabel="MAE, en consultorios",
       title="El error no es igual en todos los tamaños municipales")
plt.tight_layout(); plt.show()
display(por_tamano)

display(resumen_revision.loc[resumen_revision["grupo"].isin(["all", "observed_1_to_5_diagnostic"]),
    ["grupo", "metodo", "n", "mae", "r2", "within_one_fraction", "mean_absolute_log_error"]])
por_tamano_metodo = pd.DataFrame([
    {"poblacion": str(nombre), "metodo": metodo, **metricas(g["observado"], g[metodo])}
    for nombre, g in oof.groupby("grupo_poblacion", observed=True) for metodo in METODOS
])
display(por_tamano_metodo[["poblacion", "metodo", "n", "mae", "r2", "mean_absolute_log_error"]])


## 13. Municipios concretos: mismos casos, cuatro respuestas
Se muestran ejemplos relevantes, incluidos municipios donde todos los métodos fallan. Los casos del grupo principal pueden cambiarse editando la lista de códigos.

Puerres y Albán explican la pequeña desventaja del modelo aplicado en ese grupo. En Funes y Río Quito, los cuatro métodos responden 1 ante un registro real de 2. El empate no significa que ninguno se equivoque.

Atrato no aparece como ejemplo con verdad observada de urgencias: allí falta la etiqueta. Su cálculo se conserva después como escenario, no como validación.


In [ ]:
CODIGOS_EJEMPLO = ["66001", "76828", "27660", "27050"]
CODIGOS_REVISION = ["52573", "52019", "52287", "27600", "25530", "66001", "76828"]
display(oof.loc[oof["code"].isin(CODIGOS_REVISION),
    ["code", "municipality", "department", "observado", *METODOS, "seleccion_interna", "grupo"]])
principal = oof.loc[grupos_evaluacion["primary_small_no_beds"]].copy()
principal["delta_error_vs_1"] = np.abs(np.log(principal["poisson_capacidades"]/principal["observado"])) - np.abs(np.log(1/principal["observado"]))
display(principal.sort_values("delta_error_vs_1", ascending=False).head(10)[
    ["code", "municipality", "observado", *METODOS, "delta_error_vs_1"]])
display(oof.nlargest(10, "error_absoluto")[
    ["code", "municipality", "observado", *METODOS, "error_absoluto"]])


## 14. Decisión de la revisión y reconstrucción del modelo aplicado
La elección interna dirigida al grupo pertinente no demostró una ventaja del ML frente a la referencia constante. **No usamos ese resultado para publicar nuevas imputaciones ni para sustituir denominadores por 1.**

Para explicar el tablero existente y estudiar su sensibilidad, reconstruimos a continuación el Poisson que ya estaba aplicado. Esto no significa volver a elegirlo ni revocar la conclusión de la revisión. Los coeficientes se comprueban contra el modelo original.


In [ ]:
errores_principales = resumen_revision.loc[resumen_revision["grupo"].eq("primary_small_no_beds")].set_index("metodo")["mean_absolute_log_error"]
mejoras = errores_principales.loc[errores_principales < errores_principales["siempre_1"]-1e-12]
print("Métodos que reducen el error principal frente a siempre 1:", mejoras.index.tolist())
print("Elección interna por grupo:", [d["seleccion"] for d in decisiones])
assert mejoras.empty
display(Markdown("**Conclusión de esta corrida:** no se demostró valor predictivo añadido en el grupo principal. "
                 "Esto no valida la existencia del servicio ni autoriza imputar 1."))

config_final = next(c for c in CANDIDATOS if c["id"] == "poisson_capacidades")
ajuste_final = entrenar(observados, config_final)  # reconstruir el escenario existente
assert ajuste_final["model"]["converged"]
np.testing.assert_allclose(ajuste_final["model"]["beta"],
                           historico["model"]["model"]["beta"], rtol=1e-5, atol=1e-5)
pre_final = ajuste_final["scaler"]
coeficientes = pd.DataFrame({
    "variable": ["Intercepto"] + NOMBRES_X, "coeficiente": ajuste_final["model"]["beta"],
    "media_entrenamiento": [np.nan] + list(pre_final["avg"]),
    "desviacion_entrenamiento": [np.nan] + list(pre_final["sd"]),
})
display(coeficientes)


## 15. Contraste opcional con una implementación independiente
Se ajusta `PoissonRegressor` de scikit-learn sobre exactamente las mismas variables estandarizadas y con \(\alpha=1\). Esta celda comprueba si ambos optimizadores convergen a predicciones compatibles.

No se presenta este contraste como una prueba ya ejecutada al escribir el cuaderno. El mensaje que aparezca al ejecutar la celda es el resultado real de la comprobación.


In [ ]:
contraste_sklearn = {"estado": "no ejecutado"}
try:
    import sklearn
    from sklearn.linear_model import PoissonRegressor
except ImportError:
    print("Contraste omitido: instala scikit-learn si deseas ejecutarlo.")
else:
    X_final = transformar(observados, pre_final, True)
    modelo_sklearn = PoissonRegressor(alpha=1., max_iter=5000, tol=1e-9)
    modelo_sklearn.fit(X_final, observados["urgencias"].to_numpy())
    propio = predecir(ajuste_final, observados, piso=False)
    independiente = modelo_sklearn.predict(X_final)
    diferencia = float(np.max(np.abs(propio-independiente)))
    np.testing.assert_allclose(propio, independiente, rtol=1e-4, atol=1e-4)
    contraste_sklearn = {"estado": "predicciones compatibles", "version": sklearn.__version__,
                        "max_diferencia_absoluta": diferencia}
    display(pd.Series(contraste_sklearn))


## 16. Reconstruir las estimaciones existentes, sin volver a autorizarlas
Esta celda reproduce el filtro histórico de publicación para explicar las 203 estimaciones que ya tenía la rama:

- IPM observado.
- Predictores dentro del rango de entrenamiento.
- Al menos 30 observados con el mismo patrón de presencia/ausencia de las otras capacidades.

**Esos controles no demostraron mejora predictiva para el grupo principal.** Se conservan aquí por trazabilidad, no como certificación de precisión. No se cambian ni vuelven a publicar denominadores. La salida reproduce el escenario experimental original y distingue las 18 abstenciones.


In [ ]:
def patron(frame):
    return [(pd.isna(a), pd.isna(b)) for a, b in zip(frame["consulta_externa"], frame["camas_generales"])]

conteos_patron = Counter(patron(observados))
X_train_crudo = variables(observados)
min_train = np.array([np.min(c[np.isfinite(c)]) for c in X_train_crudo.T])
max_train = np.array([np.max(c[np.isfinite(c)]) for c in X_train_crudo.T])
X_missing = variables(faltantes)
fuera = np.isfinite(X_missing) & ((X_missing < min_train) | (X_missing > max_train))

estimaciones = faltantes[["code", "municipality", "department", "ipm_2018"]].copy()
estimaciones["prediccion_sin_piso"] = predecir(ajuste_final, faltantes, piso=False)
estimaciones["prediccion_con_piso"] = predecir(ajuste_final, faltantes)
estimaciones["n_mismo_patron"] = [conteos_patron[p] for p in patron(faltantes)]
estimaciones["fuera_de_rango"] = fuera.any(axis=1)
estimaciones["soporte"] = (estimaciones["n_mismo_patron"].ge(30)
                           & ~estimaciones["fuera_de_rango"]
                           & estimaciones["ipm_2018"].notna())
estimaciones["estimacion_publicable"] = estimaciones["prediccion_con_piso"].where(estimaciones["soporte"])
assert estimaciones["soporte"].sum() == 203
assert (~estimaciones["soporte"]).sum() == 18

guardadas = pd.DataFrame(historico["predictions"])
chequeo_estimaciones = estimaciones.merge(
    guardadas[["code", "predicted", "exploratory_supported"]], on="code", validate="one_to_one")
np.testing.assert_allclose(chequeo_estimaciones["prediccion_con_piso"],
                           chequeo_estimaciones["predicted"], rtol=1e-5, atol=1e-5)
assert chequeo_estimaciones["soporte"].eq(chequeo_estimaciones["exploratory_supported"]).all()
display(estimaciones.loc[estimaciones["code"].isin(CODIGOS_EJEMPLO)])
display(estimaciones.loc[~estimaciones["soporte"]])

aplicadas = estimaciones.loc[estimaciones["soporte"]]
assert int(aplicadas["prediccion_con_piso"].eq(1).sum()) == 174
display(pd.Series({"estimaciones_aplicadas": len(aplicadas),
                   "aplicadas_en_piso_1": int(aplicadas["prediccion_con_piso"].eq(1).sum()),
                   "aplicadas_mayores_que_1": int(aplicadas["prediccion_con_piso"].gt(1).sum())}))


## 17. El piso 1 en los casos realmente aplicados
De 221 predicciones brutas, 192 quedan en el piso. Entre las 203 elegibles, **174 quedan exactamente en 1 y sólo 29 son mayores que 1**. Esta distinción faltaba en el primer plano del cuaderno.

La restricción se impone después del ajuste. No confirma que exista al menos un consultorio; una media antes del recorte menor que 1 tampoco confirma capacidad cero. La ventaja de cualquier modelo debe evaluarse con el mismo recorte que se utiliza después.

El gráfico compara la salida antes y después del piso; la tabla anterior cuantifica las estimaciones aplicadas.


In [ ]:
n_piso = int(estimaciones["prediccion_con_piso"].eq(1).sum())
assert n_piso == 192
print(f"{n_piso}/{len(estimaciones)} predicciones brutas quedan en el piso 1.")
fig, ax = plt.subplots()
ax.scatter(estimaciones["prediccion_sin_piso"], estimaciones["prediccion_con_piso"],
           c=np.where(estimaciones["soporte"], AZUL, NARANJA), alpha=.55)
max_graf = max(1., estimaciones["prediccion_con_piso"].max())
ax.plot([0, max_graf], [0, max_graf], "--", color="gray", label="Sin recorte")
ax.axhline(1, color=NARANJA, linewidth=1, label="Piso del escenario")
ax.set(xlabel="Predicción antes del piso", ylabel="Predicción después del piso",
       title="Efecto explícito de la restricción a capacidad positiva")
ax.legend(); plt.tight_layout(); plt.show()


## 18. Cargar los daños que realmente utiliza el tablero
El `index.html` del commit congelado contiene el inventario depurado, sus códigos municipales y las reglas territoriales ya aplicadas. Leemos únicamente el objeto JSON `DATA`: **no ejecutamos JavaScript del HTML**.

Esto evita inventar un daño para el ejemplo o resolver nombres municipales con coincidencias aproximadas. Se usa exclusivamente el indicador `pnud_csalud`, fuente `PNUD`, unidad `Número`, del mismo corte. No se rellenan sus ausencias con 3iS.

El ámbito puede ser `"decree"` (departamentos del decreto) o `"all"` (todo el inventario). El máximo se calcula de nuevo para ese ámbito.


In [ ]:
html_tablero = leer_version("index.html").decode("utf-8")
marcador = re.search(r"\bconst\s+DATA\s*=\s*", html_tablero)
if marcador is None:
    raise ValueError("No se encontró el JSON DATA del tablero.")
payload, _ = json.JSONDecoder().raw_decode(html_tablero[marcador.end():])
CAPTURA = "2026-09-11"
AMBITO = "decree"  # Cambiar a "all" para todo el inventario.
assert AMBITO in {"decree", "all"} and payload["latest"] == CAPTURA

inventario = pd.DataFrame(payload["rows"])
actual = inventario.loc[inventario["date"].eq(CAPTURA)].copy()
departamentos_decreto = set(actual.loc[
    actual["id"].eq("en_decreto_1171") & actual["f"].eq("Decreto1171") & actual["v"].eq(1), "d"
])
municipales = actual.loc[actual["lv"].eq("municipal")].copy()
if AMBITO == "decree":
    municipales = municipales.loc[municipales["d"].isin(departamentos_decreto)]
territorios = municipales[["geo", "code", "m", "d"]].drop_duplicates("geo")

salud_pnud = municipales.loc[
    municipales["id"].eq("pnud_csalud") & municipales["f"].eq("PNUD")
    & municipales["u"].eq("Número"), ["geo", "v"]
].rename(columns={"v": "afectados"})
assert salud_pnud["geo"].is_unique
assert salud_pnud["afectados"].ge(0).all()

# Verificar que el registro municipal de entrenamiento coincide con los denominadores del tablero.
den_tab = pd.DataFrame(payload["denominators"]["rows"])
den_observados = den_tab.loc[
    den_tab["kind"].eq("consultorios_urgencias_reps")
    & den_tab["status"].eq("verified_historical"), ["code", "value"]
]
assert len(den_observados) == 901 and den_observados["code"].is_unique
ver_capacidad = observados.merge(den_observados, on="code", validate="one_to_one")
assert len(ver_capacidad) == 901
np.testing.assert_allclose(ver_capacidad["urgencias"], ver_capacidad["value"], rtol=0, atol=0)
print(f"{len(territorios)} territorios de referencia; ámbito={AMBITO}; captura={CAPTURA}.")
display(salud_pnud.head())


## 19. Del denominador al puntaje de Salud
\[
q_m = \frac{A_m}{B_m},\qquad S_m=100\frac{q_m}{\max_{j\in R}(q_j)}.
\]

- \(A_m\): centros afectados PNUD; falta → no se calcula.
- \(B_m\): consultorios observados; sólo cuando faltan, estimación elegible.
- \(R\): municipios comparables del ámbito y corte seleccionado.
- Daño cero explícito con denominador utilizable → presión y puntaje cero.
- Sin ambos componentes no hay dato relativo.

**No es porcentaje de consultorios destruidos:** el numerador son centros/puntos afectados y el denominador consultorios. Es presión relativa sobre una capacidad distinta.

Se muestran dos escenarios: urgencias observadas y urgencias con ML. No se reemplazan los registros existentes.


In [ ]:
panel = territorios.merge(salud_pnud, on="geo", how="left", validate="one_to_one")
panel = panel.merge(df[["code", "urgencias", "ipm_2018"]], on="code",
                    how="left", validate="many_to_one")
panel = panel.merge(estimaciones[["code", "estimacion_publicable"]], on="code",
                    how="left", validate="many_to_one")
panel["base_observada"] = panel["urgencias"]
panel["base_escenario"] = panel["base_observada"].fillna(panel["estimacion_publicable"])
panel["base_es_ML"] = panel["base_observada"].isna() & panel["estimacion_publicable"].notna()
anclas = {}
for sufijo, base in [("observado", "base_observada"), ("ml", "base_escenario")]:
    panel[f"presion_{sufijo}"] = panel["afectados"] / panel[base]
    maximo = panel[f"presion_{sufijo}"].max()
    if not np.isfinite(maximo) or maximo <= 0:
        raise ValueError("No hay un máximo positivo para normalizar este ámbito.")
    anclas[sufijo] = float(maximo)
    panel[f"salud_{sufijo}"] = (100*panel[f"presion_{sufijo}"]/maximo).clip(0, 100)

resumen_aplicacion = {
    "territorios": len(panel),
    "Salud calculable sin ML": int(panel["salud_observado"].notna().sum()),
    "Salud calculable con ML": int(panel["salud_ml"].notna().sum()),
    "Salud calculada usando ML": int((panel["base_es_ML"] & panel["salud_ml"].notna()).sum()),
    "máximo sin ML": anclas["observado"], "máximo con ML": anclas["ml"],
}
display(pd.Series(resumen_aplicacion))

auditoria_aplicacion = json.loads(leer_version("experimentos/ml_salud/impacto_aplicacion_urgencias.json"))
reporte = next(r for r in auditoria_aplicacion
               if r["scope"] == AMBITO and r["severity"] == "total" and r["date"] == CAPTURA)
assert resumen_aplicacion["territorios"] == reporte["stats"]["ml"]["reference"]
assert resumen_aplicacion["Salud calculable sin ML"] == reporte["stats"]["urgencias"]["health"]
assert resumen_aplicacion["Salud calculable con ML"] == reporte["stats"]["ml"]["health"]
assert resumen_aplicacion["Salud calculada usando ML"] == reporte["stats"]["ml"]["imputed"]
np.testing.assert_allclose(anclas["ml"], reporte["stats"]["ml"]["anchor"], rtol=1e-5)

# Comparación por municipio: el cálculo nuevo debe corresponder al puntaje de la rama.
salud_guardada = pd.DataFrame([
    {"code": r["code"], "limite_salud_archivado": r["ml"]["health"]} for r in reporte["changes"]
])
comparar_salud = panel.loc[panel["code"].ne("")].merge(salud_guardada, on="code", validate="one_to_one")
# El archivo guarda el límite documentado: un sector sin dato tiene límite 0.
# Sólo para comparar límites se usa fillna(0); panel.salud_ml conserva NaN.
np.testing.assert_allclose(comparar_salud["salud_ml"].fillna(0), comparar_salud["limite_salud_archivado"],
                           rtol=1e-5, atol=1e-5, equal_nan=True)
display(panel.loc[panel["code"].isin(CODIGOS_EJEMPLO),
                  ["code", "m", "afectados", "base_observada", "estimacion_publicable",
                   "base_escenario", "presion_ml", "salud_ml", "base_es_ML"]])


## 20. Atrato: una explicación que se puede presentar
Para el corte congelado, PNUD reporta 9 centros afectados; el escenario ML aporta un denominador de 1 consultorio de urgencias. Con máximo 16, Salud es \(100\times(9/1)/16=56,25\).

El índice global existente da peso \(1/6\) a Salud y luego aplica el factor IPM:
\[
P_m=D_m\frac{1+0,25\,IPM_m/100}{1,25}.
\]
Como el máximo no cambió en este experimento y las otras dimensiones permanecen iguales, podemos aislar el cambio de Salud:
\[
\Delta P_m=\frac{S_m^{ML}-S_m^{sin\,ML}}6
 \frac{1+0,25\,IPM_m/100}{1,25}.
\]

**En el límite documentado del índice**, un campo ausente aporta cero; eso no convierte el daño o la capacidad ausente en cero observado. La siguiente celda utiliza esa convención sólo al calcular la diferencia entre límites del escenario. No reconstruye las otras cinco dimensiones.


In [ ]:
atrato = panel.loc[panel["code"].eq("27050")].iloc[0]
atrato_reporte = next(r for r in reporte["changes"] if r["code"] == "27050")
factor_ipm = (1 + .25*atrato["ipm_2018"]/100) / 1.25
salud_antes = 0. if pd.isna(atrato["salud_observado"]) else atrato["salud_observado"]
delta_p = (atrato["salud_ml"]-salud_antes)/6 * factor_ipm
np.testing.assert_allclose(delta_p, atrato_reporte["delta_score"], rtol=1e-5)
display(pd.Series({
    "Centros afectados PNUD": atrato["afectados"],
    "Consultorios observados": atrato["base_observada"],
    "Consultorios del escenario ML": atrato["base_escenario"],
    "Presión": atrato["presion_ml"],
    "Máximo del ámbito": anclas["ml"],
    "Puntaje Salud": atrato["salud_ml"],
    "IPM 2018": atrato["ipm_2018"],
    "Factor IPM": factor_ipm,
    "Incremento del puntaje global": delta_p,
    "Global urgencias sin ML (archivo)": atrato_reporte["urgencias"]["score"],
    "Global urgencias con ML (archivo)": atrato_reporte["ml"]["score"],
    "Puesto sin ML (archivo)": atrato_reporte["urgencias"]["rank"],
    "Puesto con ML (archivo)": atrato_reporte["ml"]["rank"],
}))
display(Markdown("**Interpretación:** el aumento depende de una base estimada en el piso 1. "
                 "El modelo no confirmó la existencia de un consultorio en Atrato."))


## 21. Sensibilidad conjunta: bases, máximo, puntajes y posiciones
Variamos hipotéticamente las bases de **Atrato y Trujillo entre 1, 2 y 3**, manteniendo todos los demás datos. Para cada combinación se recalcula el máximo del ámbito, Salud de todos los municipios y su efecto en el puntaje global.

Estos valores alternativos **no son registros nuevos, correcciones de los observados ni intervalos de confianza**. El dato de Trujillo está registrado en REPS 2022; esta prueba estudia sensibilidad y no demuestra que sea erróneo.

Puede aparecer un tercer municipio como ancla: Acandí. Por eso no basta con dividir manualmente la ancla antigua ni limitar el cambio a Atrato.

La corrida JavaScript volvió a ejecutar el motor completo de seis dimensiones. Python reproduce el cambio con una identidad equivalente: conserva el puntaje global original y añade \((S_{\text{nuevo}}-S_{\text{anterior}})/6\) por el factor IPM. No modifica otras dimensiones ni coberturas. Todas las cifras y posiciones se contrastan con la corrida completa.

La matriz se actualiza al cambiar `AMBITO` y ejecutar desde el paso 18. Un escenario base 1/1 debe reproducir exactamente la rama existente.


In [ ]:
base_global = pd.DataFrame([
    {"code": r["code"], "global_antes": r["ml"]["score"], "puesto_antes": r["ml"]["rank"]}
    for r in reporte["changes"]
])

def puestos_compatibles(frame, columna):
    orden = frame.sort_values(columna, ascending=False, kind="stable")
    anterior, rango = None, 0
    salida = pd.Series(index=frame.index, dtype=float)
    for i, (idx, fila) in enumerate(orden.iterrows()):
        valor = fila[columna]
        if i == 0 or abs(valor-anterior) >= 1e-8:
            rango = i+1
        salida.loc[idx] = rango
        anterior = valor
    return salida.astype(int)

escenarios, detalle_escenarios = [], []
for base_atrato in [1., 2., 3.]:
    for base_trujillo in [1., 2., 3.]:
        p = panel.copy()
        p.loc[p["code"].eq("27050"), "base_escenario"] = base_atrato
        p.loc[p["code"].eq("76828"), "base_escenario"] = base_trujillo
        p["presion_nueva"] = p["afectados"] / p["base_escenario"]
        ancla_nueva = p["presion_nueva"].max()
        p["salud_nueva"] = 100*p["presion_nueva"]/ancla_nueva
        nombres_ancla = p.loc[p["presion_nueva"].eq(ancla_nueva), "m"].tolist()
        calculo = base_global.merge(p.loc[p["code"].ne(""),
            ["code", "m", "salud_ml", "salud_nueva", "ipm_2018"]], on="code", validate="one_to_one")
        assert calculo["ipm_2018"].notna().all()
        # Sólo el límite documentado trata un sector sin datos como aporte 0.
        # Los valores de Salud faltantes siguen siendo NaN.
        calculo["cambio_salud"] = calculo["salud_nueva"].fillna(0)-calculo["salud_ml"].fillna(0)
        calculo["global_nuevo"] = calculo["global_antes"] + calculo["cambio_salud"]/6*(1+.25*calculo["ipm_2018"]/100)/1.25
        calculo["puesto_nuevo"] = puestos_compatibles(calculo, "global_nuevo")
        a = calculo.loc[calculo["code"].eq("27050")].iloc[0]
        escenario = {
            "base_atrato": base_atrato, "base_trujillo": base_trujillo,
            "maximo": ancla_nueva, "municipios_ancla": ", ".join(nombres_ancla),
            "salud_atrato": a["salud_nueva"], "global_atrato": a["global_nuevo"],
            "puesto_atrato": int(a["puesto_nuevo"]),
            "municipios_cambia_salud": int(calculo["cambio_salud"].abs().gt(1e-9).sum()),
            "municipios_cambia_puesto": int(calculo["puesto_nuevo"].ne(calculo["puesto_antes"]).sum())
        }
        ref = next(r for r in sensibilidad_archivada["scenarios"]
                   if r["scope"] == AMBITO and r["base_atrato"] == base_atrato
                   and r["base_trujillo"] == base_trujillo)
        chequeo = calculo.merge(pd.DataFrame(ref["items"])[["code", "health", "global_score", "rank"]],
                               on="code", validate="one_to_one")
        np.testing.assert_allclose(chequeo["global_nuevo"], chequeo["global_score"], rtol=1e-5, atol=1e-6)
        np.testing.assert_allclose(chequeo["salud_nueva"], chequeo["health"], rtol=1e-5, atol=1e-5, equal_nan=True)
        assert chequeo["puesto_nuevo"].eq(chequeo["rank"]).all()
        assert escenario["municipios_cambia_salud"] == ref["changed_health"]
        assert escenario["municipios_cambia_puesto"] == ref["changed_rank"]
        escenarios.append(escenario)
        detalle_escenarios.append(calculo.assign(base_atrato=base_atrato, base_trujillo=base_trujillo))

tabla_sensibilidad = pd.DataFrame(escenarios)
display(tabla_sensibilidad)
matriz_salud = tabla_sensibilidad.pivot(index="base_trujillo", columns="base_atrato", values="salud_atrato")
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(matriz_salud.to_numpy(), vmin=0, vmax=100, cmap="Blues")
for i in range(len(matriz_salud.index)):
    for j in range(len(matriz_salud.columns)):
        ax.text(j, i, f"{matriz_salud.iloc[i,j]:.2f}", ha="center", va="center",
                color="white" if matriz_salud.iloc[i,j] > 65 else "black")
ax.set_xticks(range(3), [str(int(x)) for x in matriz_salud.columns])
ax.set_yticks(range(3), [str(int(x)) for x in matriz_salud.index])
ax.set(xlabel="Base hipotética de Atrato", ylabel="Base hipotética de Trujillo",
       title="Salud de Atrato: máximo recalculado en cada escenario")
fig.colorbar(im, ax=ax, label="Salud / 100")
plt.tight_layout(); plt.show()

display(pd.concat(detalle_escenarios).loc[
    lambda x: x["base_atrato"].eq(1) & x["base_trujillo"].eq(2)
].sort_values("puesto_nuevo").head(15)[
    ["code", "m", "salud_ml", "salud_nueva", "global_antes", "global_nuevo", "puesto_antes", "puesto_nuevo"]
])


## 22. Cambios del ranking y la comparación con UNGRD
El archivo de auditoría contiene el cálculo completo de las seis dimensiones para cada escenario. Aquí se lee ese resultado **archivado**; no se presenta como si este cuaderno hubiera recalculado las demás dimensiones.

Se comparan urgencias sin ML y urgencias con ML. El cambio desde consulta externa también altera el denominador; no hay que confundir ambos efectos.

R² con UNGRD es asociación entre índices, **no validación del número de consultorios** ni evidencia de causalidad. El modelo no fue entrenado para maximizarlo.


In [ ]:
ranking_archivado = pd.DataFrame([
    {"municipio": r["m"], "departamento": r["d"],
     "puntaje_sin_ML": r["urgencias"]["score"], "puntaje_con_ML": r["ml"]["score"],
     "puesto_sin_ML": r["urgencias"]["rank"], "puesto_con_ML": r["ml"]["rank"],
     "cambio_puntaje": r["delta_score"], "campos_estimados": r["ml"]["estimated"]}
    for r in reporte["changes"]
])
display(ranking_archivado.sort_values("puesto_con_ML").head(15))
display(pd.DataFrame([
    {"escenario": nombre, "pares_UNGRD": reporte["stats"][clave]["n"],
     "R2_con_UNGRD": reporte["stats"][clave]["regression"]["r2"],
     "Spearman_con_UNGRD": reporte["stats"][clave]["rho"]}
    for clave, nombre in [("consulta", "Consulta externa observada"),
                          ("urgencias", "Urgencias observadas"), ("ml", "Urgencias con ML")]
]))


## 23. Conclusión corregida para presentar

**Resultado de la prueba:**
> En los municipios observados que se parecen al grupo mayoritario donde imputamos —pequeños y sin registro de camas—, no demostramos una mejora predictiva frente a responder siempre 1. La mediana de comparables empata; retirar las capacidades y usar sólo demografía no mejora el resultado. El modelo aplicado no debe justificarse con su R² nacional como si representara el uso en esos municipios.

Esto **no** demuestra que la constante 1 sea correcta donde no hay registro. El experimento aprende sobre capacidad positiva observada; no distingue ausencia del servicio de falta de información. Tampoco demuestra una inferioridad estadística general del ML: la pequeña diferencia del grupo principal se concentra en dos municipios de un departamento.

**Qué cambia en la interpretación:**
- Los 901 observados, los 221 faltantes y los 203 casos aplicados son poblaciones diferentes.
- El grupo principal tiene 69 observados de 11 departamentos frente a 146 casos aplicados. Hay muy poca información en algunos grupos de validación.
- El R² nacional se conserva como contexto; no es una tasa de aciertos ni justificación suficiente para imputar.
- La presión daño/base sigue comparando unidades distintas.
- Capacidad 2022, IPM 2018 y población 2026 no acreditan operación actual.
- El IPM influye indirectamente en el denominador estimado y directamente en el ajuste global.
- El máximo puede cambiar de municipio; un error local puede mover muchos puntajes.
- Las bandas bootstrap son descriptivas de esta muestra reutilizada, no pruebas externas ni intervalos para cada municipio.
- Los límites de faltantes del tablero no incorporan incertidumbre del ML.
- No se cambiaron el tablero, los denominadores ni los rankings publicados.

**Siguiente decisión:** mantener estas imputaciones identificadas como escenario experimental, sin presentarlas como mejora validada. Una muestra externa de municipios hoy sin registro —incluidos ceros verificables— permitiría una evaluación más decisiva. Esta revisión no promete que un modelo más complejo resuelva el problema.


## 24. Exportación opcional, sin tocar el tablero
Por defecto no se escribe ningún archivo. Si se activa la exportación, se crea una carpeta nueva de resultados del notebook; no se sobrescribe una carpeta existente. Esto no publica estimaciones como registros oficiales ni regenera `index.html`.


In [ ]:
EXPORTAR = False
if EXPORTAR:
    from datetime import datetime
    destino = Path.cwd() / ("revision_notebook_salud_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    destino.mkdir(exist_ok=False)
    oof.to_csv(destino / "validacion_cuatro_metodos.csv", index=False)
    resumen_revision.to_csv(destino / "metricas_por_grupo.csv", index=False)
    estabilidad.to_csv(destino / "estabilidad_departamentos.csv", index=False)
    tabla_sensibilidad.to_csv(destino / "sensibilidad_conjunta.csv", index=False)
    pd.concat(detalle_escenarios).to_csv(destino / "cambios_municipales_sensibilidad.csv", index=False)
    pd.DataFrame(trazabilidad).to_csv(destino / "trazabilidad.csv", index=False)
    print(f"Resultados guardados en: {destino.resolve()}")
else:
    print("Sin escritura de archivos. No se ha modificado el tablero.")


## Referencias y archivos para acompañar la presentación

- [Resultados completos de la revisión](https://github.com/Practicantepotencia/tablero-terremoto/blob/01cfb4916e52357f61db4eba3c961cf8fa7eeaf2/experimentos/ml_salud/revision_faltantes/comparacion.json).
- [Código de la revisión dirigida](https://github.com/Practicantepotencia/tablero-terremoto/blob/01cfb4916e52357f61db4eba3c961cf8fa7eeaf2/scripts/revisar_ml_salud.cjs).
- [Sensibilidad conjunta calculada con el motor original](https://github.com/Practicantepotencia/tablero-terremoto/blob/01cfb4916e52357f61db4eba3c961cf8fa7eeaf2/experimentos/ml_salud/revision_faltantes/sensibilidad.json).

- [Datos municipales del experimento](https://github.com/Practicantepotencia/tablero-terremoto/blob/a350ca6591174542e76b08d108e4d4e15c629b4c/experimentos/ml_salud/entrada.json).
- [Método original y resultados](https://github.com/Practicantepotencia/tablero-terremoto/blob/a350ca6591174542e76b08d108e4d4e15c629b4c/docs/ML_SALUD.md).
- [Código JavaScript original](https://github.com/Practicantepotencia/tablero-terremoto/blob/a350ca6591174542e76b08d108e4d4e15c629b4c/scripts/ml_salud.cjs).
- [Aplicación experimental al tablero](https://github.com/Practicantepotencia/tablero-terremoto/blob/a350ca6591174542e76b08d108e4d4e15c629b4c/docs/ML_URGENCIAS_APLICADO.md).
- [REPS — capacidad instalada, fuente oficial](https://www.datos.gov.co/Salud-y-Protecci-n-Social/Relaci-n-de-IPS-p-blicas-y-privadas-seg-n-el-nivel/s2ru-bqt6/about_data).
- [DANE — proyecciones de población](https://www.dane.gov.co/index.php/estadisticas-por-tema/demografia-y-poblacion/proyecciones-de-poblacion).
- [DANE — IPM municipal censal 2018](https://www.dane.gov.co/files/investigaciones/condiciones_vida/pobreza/2018/informacion-censal/anexo-censal-pobreza-municipal-2018.xlsx).
- [NumPy: solución del sistema de Newton](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html).
- [scikit-learn: PoissonRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.PoissonRegressor.html).

**Trazabilidad de la autoría:** notebook preparado el 15 de septiembre de 2026. Python no pudo ejecutarse en el entorno de creación; no se incluyen salidas aparentando una ejecución. La nueva revisión se ejecutó en JavaScript V8: 901 predicciones por método, selección dirigida y 18 escenarios con el motor real. Las nuevas celdas Python no se ejecutaron aquí; contienen comprobaciones contra esos resultados. La reproducción del notebook anterior comunicada por el usuario no se atribuye a esta nueva versión.
